In [3]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

In [4]:
nltk.download('stopwords')

# 1. Load your data
file_path = "movie_reviews.csv"  # replace with your actual file path
df = pd.read_csv(file_path)

# 2. Filter to keep only positive and negative sentiments
df = df[df['sentiment'].isin(['positive', 'negative'])]

# 3. Text cleaning function
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)            # remove URLs
    text = re.sub(r'[^a-z0-9\s]', ' ', text)       # remove punctuation
    tokens = text.split()
    tokens = [word for word in tokens if word not in stopwords.words('english')]
    return " ".join(tokens)

df['cleaned_review'] = df['review'].apply(clean_text)

# 4. Encode labels (positive → 1, negative → 0)
df['label_encoded'] = np.where(df['sentiment'] == 'positive', 1, 0)

# 5. Split data into training and testing sets
X = df['cleaned_review']
y = df['label_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/jarvis/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))),
    ('clf', LogisticRegression(solver='liblinear', C=1.0, random_state=42))
])

# 7. Train model
pipeline.fit(X_train, y_train)


Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
                ('clf',
                 LogisticRegression(random_state=42, solver='liblinear'))])

In [6]:
y_pred = pipeline.predict(X_test)
print("✅ Accuracy:", accuracy_score(y_test, y_pred))
print("\n📊 Classification Report:\n", classification_report(y_test, y_pred, target_names=['negative', 'positive']))
print("\n🧩 Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

✅ Accuracy: 0.82

📊 Classification Report:
               precision    recall  f1-score   support

    negative       0.80      0.88      0.84       105
    positive       0.85      0.76      0.80        95

    accuracy                           0.82       200
   macro avg       0.82      0.82      0.82       200
weighted avg       0.82      0.82      0.82       200


🧩 Confusion Matrix:
 [[92 13]
 [23 72]]


In [7]:
new_reviews = [
    "This movie was amazing and full of excitement!",
    "I regretted going to watch it, very disappointing."
]

# Clean and predict
new_cleaned = [clean_text(r) for r in new_reviews]
preds = pipeline.predict(new_cleaned)

for review, pred in zip(new_reviews, preds):
    label = 'positive' if pred == 1 else 'negative'
    print(f"\n🎬 Review: {review}\n→ Sentiment: {label}")

# 10. Save the trained model
joblib.dump(pipeline, 'binary_sentiment_model.pkl')
print("\n💾 Model saved as binary_sentiment_model.pkl")


🎬 Review: This movie was amazing and full of excitement!
→ Sentiment: positive

🎬 Review: I regretted going to watch it, very disappointing.
→ Sentiment: negative

💾 Model saved as binary_sentiment_model.pkl
